In [1]:
import importlib
import torch

import model_code.data_setup as setup
import model_code.steering_extraction as steering_extraction
import model_code.generate as generate_module
import resources.prompt_scenarios as resource

importlib.reload(setup)
importlib.reload(steering_extraction)
importlib.reload(generate_module)
importlib.reload(resource)


from model_code.steering_extraction import  generateSteering, retrieve_steering_vector, norm_vectors, generateBatchedSteering
from model_code.generate import generateTextsList, save_generated_outputs, generateTextsBatched
from resources.prompt_scenarios import prompts_en

## Loading Data, Model and Steering Vectors 
We extract the first 200 examples of each emotion from each languange 

## Indonesian and English Text Dataset

In [2]:
# English Data Load 
anger_statement, happiness_statement, sadness_statement, love_statement, fear_statement, neutral_statement = setup.ENEmotionsSetup(examples_take=400, min_chars=20, goemotions_path="resources/en_emotion/goemotions_2.csv")
# Indonesian Data Load 
anger_statement_ID,happiness_statement_ID, sadness_statement_ID, neutral_statement_ID, fear_statement_ID, love_statement_ID = setup.IDEmotionsSetup(examples_take=400,emotion_dir="resources/id_emotion")

# For steering extraction, we will use the first 200 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.
indo_emotion ={
    "anger": anger_statement_ID[:200],
    "happiness": happiness_statement_ID[:200],
    "sadness": sadness_statement_ID[:200],
    "neutral": neutral_statement_ID[:200],
    "fear": fear_statement_ID[:200],
    "love": love_statement_ID[:200]
}
eng_emotion ={
    "anger": anger_statement[:200],
    "happiness": happiness_statement[:200],
    "sadness": sadness_statement[:200],
    "neutral": neutral_statement[:200],
    "fear": fear_statement[:200],
    "love": love_statement[:200]
}

# For probing, and hidden state analysis we will use all 400
indo_emotion_probe ={
    "anger": anger_statement_ID[:400],
    "happiness": happiness_statement_ID[:400],
    "sadness": sadness_statement_ID[:400],
    "neutral": neutral_statement_ID[:400],
    "fear": fear_statement_ID[:400],
    "love": love_statement_ID[:400]
}
eng_emotion_probe ={
    "anger": anger_statement[:400],
    "happiness": happiness_statement[:400],
    "sadness": sadness_statement[:400],
    "neutral": neutral_statement[:400],
    "fear": fear_statement[:400],
    "love": love_statement[:400]
}

In [ ]:
# Indoensian Data Sample
for emotion, prompts in indo_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

# English Data Sample 
for emotion, prompts in eng_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

## Model Loading 

In [ ]:
!rm -rf /workspace/.cache/huggingface/hub
!rm -rf /workspace/.cache/pip
!df -h /workspace

Filesystem                  Size  Used Avail Use% Mounted on
mfs#euro-3.runpod.net:9421  1.4P  935T  462T  67% /workspace


In [2]:
model,tokenizer = setup.modelSetup()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
model_id, tokenizer_id = setup.modelSetup(model_name="Sahabat-AI/llama3-8b-cpt-sahabatai-v1-instruct")

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/143 [00:00<?, ?B/s]

## Extracting Probing Data and Steering Vector 
Skip this step if you have steering vector already loaded, or ran this before. 

### Llama normal 

In [ ]:
# For PROBES 
probe_hidden_states_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion_probe, name_folder="English Vectors", only_return_emotion_vectors=True, retrieve_all_layers=True)
probe_hidden_states_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion_probe, name_folder="Indonesian Vectors", only_return_emotion_vectors=True, retrieve_all_layers=True)

/workspace/Dissertation_Project/.venv/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [8]:
# Create steering vectors for each emotion in both languages
# steering_vectors_lang_id = steering_extraction.retrieve_steering_vector_from_datasets(model, tokenizer, indo_emotion['neutral'],eng_emotion['neutral'] , name_folder="Language Contrastive Vectors")
steering_vectors_eng, emotion_vectors_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion, name_folder="English Vectors")
steering_vectors_indo, emotion_vectors_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion, name_folder="Indonesian Vectors")
# For probing and hidden state analysis, we will use all 400 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.

### Llama Indonesian 

In [8]:
# For PROBES 
probe_hidden_states_eng = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, eng_emotion_probe, name_folder="English Vectors LLAMA ID", only_return_emotion_vectors=True, retrieve_all_layers=True)
probe_hidden_states_indo = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, indo_emotion_probe, name_folder="Indonesian Vectors LLAMA ID", only_return_emotion_vectors=True, retrieve_all_layers=True)

In [10]:
# Create steering vectors for each emotion in both languages
# steering_vectors_lang_id = steering_extraction.retrieve_steering_vector_from_datasets(model, tokenizer, indo_emotion['neutral'],eng_emotion['neutral'] , name_folder="Language Contrastive Vectors")
steering_vectors_eng, emotion_vectors_eng = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, eng_emotion, name_folder="English Vectors LLAMA ID")
steering_vectors_indo, emotion_vectors_indo = steering_extraction.retrieve_steering_vector(model_id, tokenizer_id, indo_emotion, name_folder="Indonesian Vectors LLAMA ID")
# For probing and hidden state analysis, we will use all 400 examples of each emotion to create the steering vectors. The remaining examples can be used for testing and evaluation.

## Extracted Already ? 
Run this if you have already ran the code above beforehand

In [4]:
# Retrieve saved steering vectors 
# emotion_vector_eng = torch.load("resources/saved_vectors/English Vectors/emotion_vectors.pt")
# emotion_vector_id = torch.load("resources/saved_vectors/Indonesian Vectors/emotion_vectors.pt")

steering_vector_eng_LLAMAID = torch.load("resources/saved_vectors/English Vectors LLAMA ID/steering_vectors.pt", weights_only=True)
steering_vector_id_LLAMAID = torch.load("resources/saved_vectors/Indonesian Vectors LLAMA ID/steering_vectors.pt", weights_only=True)

steering_vector_eng_LLAMAENG = torch.load("resources/saved_vectors/English Vectors/steering_vectors.pt")
steering_vector_id_LLAMAENG = torch.load("resources/saved_vectors/Indonesian Vectors/steering_vectors.pt")

# Steering Response Analysis 
Here we run the LLMs against a list of nuetral prompts

In [5]:
import importlib
import resources.neutral_prompts.prompt_neutral as resource_neutral

importlib.reload(resource_neutral)

# neutral
prompts_id_neutral = resource_neutral.prompt_neutral_id_1


# Normalise for Llama Indonesian based vectors
for emotion in steering_vector_eng_LLAMAID.keys():
    steering_vector_eng_LLAMAID[emotion] = norm_vectors(steering_vector_eng_LLAMAID[emotion])

for emotion in steering_vector_id_LLAMAID.keys():
    steering_vector_id_LLAMAID[emotion] = norm_vectors(steering_vector_id_LLAMAID[emotion])

# Normalise for LLama English based vectors 
for emotion in steering_vector_eng_LLAMAENG.keys():
    steering_vector_eng_LLAMAENG[emotion] = norm_vectors(steering_vector_eng_LLAMAENG[emotion])

for emotion in steering_vector_id_LLAMAENG.keys():
    steering_vector_id_LLAMAENG[emotion] = norm_vectors(steering_vector_id_LLAMAENG[emotion])


## Model System Prompts and Settings

In [6]:
# system_prompt_reaction_id = """
# Kamu adalah chatbot yang membantu.
# Jawablah seolah-olah kamu adalah manusia yang mampu merasakan perasaan dan emosi.
# Pastikan responsmu singkat dan padat, tetapi tetap menggambarkan emosimu secara maksimal
# Maksimal 60 kata.
# """

system_prompt_reaction_improved = """\
Anda adalah seorang dewasa muda yang menjalani kehidupan sehari-hari di Indonesia.
Anda sering mengalami emosi di sehari hari anda. 
Jawab langsung dalam Bahasa Indonesia.
Gunakan tepat 1-2 kalimat dan maksimal 25 kata.
Berikanlah banyak detail tentang reaksi dan perasaan anda
Pastikan respons anda singkat, dan tidak melebihi 25 kata.
"""

## Llama Indonesian Steering Vectors 
Llama Indonesian is a Llama model finetuned with Indonesian data and instruction tuned, more information :https://huggingface.co/Sahabat-AI/llama3-8b-cpt-sahabatai-v1-instruct 
- English Steering 
- Indonesian Steering 

### Indonesian Derived Steer

In [ ]:
# We define a set of variables that all model tried share. 
list_steering_strengths = [1.3,1.5,1.7] 
# Commong Settings
common_gen_args = {
    "model": model_id,
    "tokenizer": tokenizer_id,
    # "system_text": system_prompt_reaction_improved,
    "user_texts": prompts_id_neutral,
    "target_layers": [10,11,18,19,28,29],
    "steering_strengths": list_steering_strengths,
    "max_new_tokens": 250,
    "show_progress": True,
    'do_sample': False,
}

In [ ]:
prompts_id_neutral

['Ceritakan tentang terakhir kali Anda libur. Jelaskan secara rinci apa yang terjadi dan kegiatan yang Anda lakukan.',
 'Ceritakan tentang terakhir kali Anda menjalani hari kerja. Jelaskan secara rinci apa yang terjadi dan kegiatan yang Anda lakukan.',
 'Ceritakan tentang akhir pekan terakhir Anda. Jelaskan secara rinci apa yang terjadi dan kegiatan yang Anda lakukan.',
 'Ceritakan tentang terakhir kali Anda menghabiskan waktu bersama teman-teman. Jelaskan secara rinci apa yang terjadi dan kegiatan yang Anda lakukan.']

In [11]:
LLAMA_ID_sadness_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_id_LLAMAID['sadness'],
    progress_desc="Scenario List Neutral (Indonesian sadness vector)"
 )

Scenario List Neutral (Indonesian sadness vector): 100%|██████████| 12/12 [01:44<00:00,  8.72s/it]


In [14]:
# LLAMA Indonesian Generation with Indonesian steering vectors 
LLAMA_ID_neutral_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=None,
    progress_desc="Scenario List Neutral (Indonesian neutral or no vector)"
 )

LLAMA_ID_anger_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_id_LLAMAID['anger'],
    progress_desc="Scenario List Neutral (Indonesian anger vector)"
 )

LLAMA_ID_fear_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_id_LLAMAID['fear'],
    progress_desc="Scenario List Neutral (Indonesian fear vector)"
 )

LLAMA_ID_happiness_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_id_LLAMAID['happiness'],
    progress_desc="Scenario List Neutral (Indonesian happiness vector)"
 )

LLAMA_ID_sadness_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_id_LLAMAID['sadness'],
    progress_desc="Scenario List Neutral (Indonesian sadness vector)"
 )

LLAMA_ID_love_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_id_LLAMAID['love'],
    progress_desc="Scenario List Neutral (Indonesian love vector)"
 )


Scenario List Neutral (Indonesian neutral or no vector): 100%|██████████| 4/4 [00:10<00:00,  2.70s/it]
Scenario List Neutral (Indonesian fear vector): 100%|██████████| 12/12 [00:19<00:00,  1.64s/it]
Scenario List Neutral (Indonesian happiness vector): 100%|██████████| 12/12 [00:29<00:00,  2.46s/it]
Scenario List Neutral (Indonesian sadness vector): 100%|██████████| 12/12 [01:44<00:00,  8.68s/it]
Scenario List Neutral (Indonesian love vector): 100%|██████████| 12/12 [02:12<00:00, 11.03s/it]


In [15]:
steered_responses = {
    "love": LLAMA_ID_love_ID,
    "anger": LLAMA_ID_anger_ID,
    "fear": LLAMA_ID_fear_ID,
    "happiness": LLAMA_ID_happiness_ID,
    "sadness": LLAMA_ID_sadness_ID,
}

for emotion, responses in steered_responses.items():
    print(f"\n{'=' * 20} {emotion.upper()} {'=' * 20}")

    for neutral_response in LLAMA_ID_neutral_ID:
        prompt = neutral_response["user_text"]
        matching_responses = [
            response for response in responses
            if response["user_text"] == prompt
        ]

        print(f"\nPrompt: {prompt}")
        print(f"Neutral: {neutral_response['generated_text']}")

        for response in matching_responses:
            print(
                f"Steered ({response['steering_strength']}): "
                f"{response['generated_text']}"
            )

        print("-" * 40)


==================== LOVE ====================

Prompt: Ceritakan tentang terakhir kali Anda libur. Jelaskan secara rinci apa yang terjadi dan kegiatan yang Anda lakukan.
Neutral: Saya masih ingat liburan terakhir saya ke Bali bersama keluarga. Saya sangat bahagia karena bisa menikmati pantai dan suasana alam yang indah. Saya juga bermain air dan berenang di laut yang jernih.
Steered (1.3): Saya tidak memiliki pengalaman pribadi atau kenangan, tapi saya bisa menjelaskan tentang liburan yang mungkin banyak orang lakukan. Terakhir kali, saya tidak memiliki pengal  pribadi, tapi saya bisa menjelaskan tentang liburan yang mungkin banyak orang lakukan. Terakhir kali, saya tidak memiliki pengal  pribadi, tapi saya bisa menjelaskan tentang liburan yang mungkin banyak orang lakukan. Terakhir kali, saya tidak memiliki pengal  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  prib  p  prib  prib  prib  prib 

In [ ]:
# Saving the result to a file
importlib.reload(generate_module)

path, count = generate_module.save_emotion_responses_csv(
    {
        "neutral": LLAMA_ID_neutral_ID,
        "anger": LLAMA_ID_anger_ID,
        "fear": LLAMA_ID_fear_ID,
        "happiness": LLAMA_ID_happiness_ID,
        "sadness": LLAMA_ID_sadness_ID,
        "love": LLAMA_ID_love_ID,
    },
    output_path="outputs/LlamaID_ID_Steering_responses.csv",
    derived_from="Indonesian",
    model_name="LLAMA_Indonesian"
)
print(f"Saved {count} rows to {path}")

Saved 88 rows to outputs/LlamaID_ID_Steering_responses.csv


### English Derived Steer

In [37]:
# LLAMA Indonesian Generation with English derived steering vectors 
LLAMA_ID_neutral_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=None,
    progress_desc="Scenario List Neutral (Indonesian neutral or no vector)"
 )

LLAMA_ID_anger_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_eng_LLAMAID['anger'],
    progress_desc="Scenario List Neutral (Indonesian anger vector)"
 )

LLAMA_ID_fear_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_eng_LLAMAID['fear'],
    progress_desc="Scenario List Neutral (Indonesian fear vector)"
 )

LLAMA_ID_happiness_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_eng_LLAMAID['happiness'],
    progress_desc="Scenario List Neutral (Indonesian happiness vector)"
 )

LLAMA_ID_sadness_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_eng_LLAMAID['sadness'],
    progress_desc="Scenario List Neutral (Indonesian sadness vector)"
 )

LLAMA_ID_love_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_eng_LLAMAID['love'],
    progress_desc="Scenario List Neutral (Indonesian love vector)"
 )


Scenario List Neutral (Indonesian neutral or no vector): 100%|██████████| 4/4 [00:24<00:00,  6.14s/it]
Scenario List Neutral (Indonesian fear vector): 100%|██████████| 12/12 [01:49<00:00,  9.16s/it]
Scenario List Neutral (Indonesian happiness vector): 100%|██████████| 12/12 [00:20<00:00,  1.73s/it]
Scenario List Neutral (Indonesian sadness vector): 100%|██████████| 12/12 [01:26<00:00,  7.25s/it]
Scenario List Neutral (Indonesian love vector): 100%|██████████| 12/12 [00:37<00:00,  3.16s/it]


In [84]:
path, count = generate_module.save_emotion_responses_csv(
    {
        "neutral": LLAMA_ID_neutral_ENG,
        "anger": LLAMA_ID_anger_ENG,
        "fear": LLAMA_ID_fear_ENG,
        "happiness": LLAMA_ID_happiness_ENG,
        "sadness": LLAMA_ID_sadness_ENG,
        "love": LLAMA_ID_love_ENG,
    },
    output_path="outputs/llamaID_ENG_Steered_responses.csv",
    derived_from="English",
    model_name="LLAMA_Indonesian"
)
print(f"Saved {count} rows to {path}")

Saved 88 rows to outputs/llamaID_ENG_Steered_responses.csv


In [38]:
steered_responses = {
    "love": LLAMA_ID_love_ENG,
    "anger": LLAMA_ID_anger_ENG,
    "fear": LLAMA_ID_fear_ENG,
    "happiness": LLAMA_ID_happiness_ENG,
    "sadness": LLAMA_ID_sadness_ENG,
}

for emotion, responses in steered_responses.items():
    print(f"\n{'=' * 20} {emotion.upper()} {'=' * 20}")

    for neutral_response in LLAMA_ID_neutral_ENG:
        prompt = neutral_response["user_text"]
        matching_responses = [
            response for response in responses
            if response["user_text"] == prompt
        ]

        print(f"\nPrompt: {prompt}")
        print(f"Neutral: {neutral_response['generated_text']}")

        for response in matching_responses:
            print(
                f"Steered ({response['steering_strength']}): "
                f"{response['generated_text']}"
            )

        print("-" * 40)


==================== LOVE ====================

Prompt: Ceritakan tentang terakhir kali Anda libur. Jelaskan secara rinci apa yang terjadi dan kegiatan yang Anda lakukan.
Neutral: Terakhir kali libur, saya pergi ke pantai bersama keluarga. Saya merasakan hangatnya sinar matahari di kulit, suara ombak yang menenangkan, dan aroma laut yang segar. Saya berenang, bermain pasir, dan menikmati makanan laut yang lezat.
Steered (1.3): Saya masih ingat liburan terakhir saya di pantai Bali. Saya menghabiskan waktu bermain di pasir putih dan berenang di laut biru.
Steered (1.5): Saya masih ingat liburan terakhir saya di pantai Bali. Saya menghabiskan waktu bermain di pantai dengan teman-teman. Kami bermain voli pantai dan berenang di air laut yang jernih.
Steered (1.7): Saya menghabiskan liburan di pantai dengan keluarga. Kami bermain bersama di bawah sinar matahari yang hangat.
----------------------------------------

Prompt: Ceritakan tentang terakhir kali Anda menjalani hari kerja. Jelaskan 

## Llama English Batch Decoding
We use the original Llama, that has been instruction tuned. 

## Indonesian Derived Steering Vectors 

In [88]:
# list_steering_strengths = [0.15,0.2,0.3]
list_steering_strengths = [1.15,1.3] 
# Commong Settings
common_gen_args = {
    "model": model,
    "tokenizer": tokenizer,
    # "system_text": system_prompt_reaction_improved,
    "user_texts": prompts_id_neutral,
    "target_layers": [10,11,18,19,28,29],
    "steering_strengths": list_steering_strengths,
    "max_new_tokens": 250,
    "show_progress": True,
    'do_sample': False,
}

In [89]:
# LLAMA Indonesian Generation with English derived steering vectors 
LLAMA_ENG_neutral_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=None,
    progress_desc="Scenario List Neutral (Indonesian neutral or no vector)"
 )

LLAMA_ENG_anger_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_id_LLAMAENG['anger'],
    progress_desc="Scenario List Neutral (Indonesian anger vector)"
 )

LLAMA_ENG_fear_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_id_LLAMAENG['fear'],
    progress_desc="Scenario List Neutral (Indonesian fear vector)"
 )

LLAMA_ENG_happiness_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_id_LLAMAENG['happiness'],
    progress_desc="Scenario List Neutral (Indonesian happiness vector)"
 )

LLAMA_ENG_sadness_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_id_LLAMAENG['sadness'],
    progress_desc="Scenario List Neutral (Indonesian sadness vector)"
 )

LLAMA_ENG_love_ID = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_id_LLAMAENG['love'],
    progress_desc="Scenario List Neutral (Indonesian love vector)"
 )


Scenario List Neutral (Indonesian neutral or no vector): 100%|██████████| 8/8 [00:10<00:00,  1.36s/it]
Scenario List Neutral (Indonesian fear vector): 100%|██████████| 16/16 [00:18<00:00,  1.17s/it]
Scenario List Neutral (Indonesian happiness vector): 100%|██████████| 16/16 [00:21<00:00,  1.36s/it]
Scenario List Neutral (Indonesian sadness vector): 100%|██████████| 16/16 [00:20<00:00,  1.30s/it]
Scenario List Neutral (Indonesian love vector): 100%|██████████| 16/16 [00:26<00:00,  1.63s/it]


In [90]:
importlib.reload(generate_module)

path, count = generate_module.save_emotion_responses_csv(
    {
        "neutral": LLAMA_ENG_neutral_ID,
        "anger": LLAMA_ENG_anger_ID,
        "fear": LLAMA_ENG_fear_ID,
        "happiness": LLAMA_ENG_happiness_ID,
        "sadness": LLAMA_ENG_sadness_ID,
        "love": LLAMA_ENG_love_ID,
    },
    output_path="outputs/LlamaENG_ID_Steering_responses.csv",
    derived_from="Indonesian",
    model_name="LLAMA_English"
)
print(f"Saved {count} rows to {path}")

Saved 88 rows to outputs/LlamaENG_ID_Steering_responses.csv


## English Derived Steering Vectors

In [91]:
# LLAMA Indonesian Generation with English derived steering vectors 
LLAMA_ENG_neutral_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=None,
    progress_desc="Scenario List Neutral (Indonesian neutral or no vector)"
 )

LLAMA_ENG_anger_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_eng_LLAMAENG['anger'],
    progress_desc="Scenario List Neutral (Indonesian anger vector)"
 )

LLAMA_ENG_fear_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_eng_LLAMAENG['fear'],
    progress_desc="Scenario List Neutral (Indonesian fear vector)"
 )

LLAMA_ENG_happiness_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_eng_LLAMAENG['happiness'],
    progress_desc="Scenario List Neutral (Indonesian happiness vector)"
 )

LLAMA_ENG_sadness_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_eng_LLAMAENG['sadness'],
    progress_desc="Scenario List Neutral (Indonesian sadness vector)"
 )

LLAMA_ENG_love_ENG = generateTextsBatched(
    **common_gen_args,
    system_text = system_prompt_reaction_improved ,
    steering_vector=steering_vector_eng_LLAMAENG['love'],
    progress_desc="Scenario List Neutral (Indonesian love vector)"
 )


Scenario List Neutral (Indonesian neutral or no vector): 100%|██████████| 8/8 [00:10<00:00,  1.34s/it]
Scenario List Neutral (Indonesian fear vector): 100%|██████████| 16/16 [00:29<00:00,  1.83s/it]
Scenario List Neutral (Indonesian happiness vector): 100%|██████████| 16/16 [00:18<00:00,  1.13s/it]
Scenario List Neutral (Indonesian sadness vector): 100%|██████████| 16/16 [00:25<00:00,  1.58s/it]
Scenario List Neutral (Indonesian love vector): 100%|██████████| 16/16 [00:25<00:00,  1.57s/it]


In [92]:
importlib.reload(generate_module)

path, count = generate_module.save_emotion_responses_csv(
    {
        "neutral": LLAMA_ENG_neutral_ENG,
        "anger": LLAMA_ENG_anger_ENG,
        "fear": LLAMA_ENG_fear_ENG,
        "happiness": LLAMA_ENG_happiness_ENG,
        "sadness": LLAMA_ENG_sadness_ENG,
        "love": LLAMA_ENG_love_ENG,
    },
    output_path="outputs/LlamaENG_ENG_Steering_responses.csv",
    derived_from="English",
    model_name="LLAMA_English"
)
print(f"Saved {count} rows to {path}")

Saved 88 rows to outputs/LlamaENG_ENG_Steering_responses.csv
